# 08c_nplikeness_filter — 천연물다움(NP-likeness)으로 합성 열거물 제거 (신규)

**한 줄 요약:** 스크리닝 후보에 **NP-likeness 점수**(Ertl 2008)를 매겨, 할로겐 필터로도 안 걸리던 **할로겐 없는 합성 조합화학 화합물**을 걸러낸다.
**왜:** COCONUT에 합성 열거물이 대량 섞여 있고(예: GW-4064 −0.64), 모델이 이들을 상위로 올린다. 구조 지표(할로겐)만으론 부족.
**NP-score:** 수많은 천연물/합성물 통계로 학습된 점수. **양수=천연물다움, 음수=합성물다움**(예: glucose +2.6, morphine +2.6, GW-4064 −0.64).
**큰 흐름:** ① 준비 → ② 후보·모델 로드 → ③ 점수 계산 → ④ 정제(확률+NP 동시 통과)

> **📌 읽는 법**: 각 코드 셀은 [① 무슨 작업] → [② 코드] → [③ 🔎 코드 뜯어보기].

### 준비 — 도구 불러오기
RDKit 부속(Contrib)의 NP-likeness 점수 모듈을 불러온다.

In [ ]:
import os, sys
while not os.path.isdir('data') and os.path.dirname(os.getcwd()) != os.getcwd():
    os.chdir('..')  # data/ 폴더를 찾을 때까지 상위로 (하위 폴더에서 열어도 동작)
print('작업 폴더:', os.getcwd())
import pandas as pd
from rdkit import Chem
from rdkit.Chem import RDConfig
from rdkit import RDLogger
RDLogger.DisableLog('rdApp.*')
# RDKit 부속(Contrib)의 NP-likeness 점수(Ertl 2008) 불러오기
sys.path.append(os.path.join(RDConfig.RDContribDir, 'NP_Score'))
import npscorer

🔎 **코드 뜯어보기 (준비)**
- `RDConfig.RDContribDir` : RDKit 부속 코드 폴더 경로. `sys.path.append(...)` 로 그 안 `NP_Score`를 import 가능하게 한 뒤 `npscorer` 불러옴.

### 셀 1 — 후보 + NP 모델 로드
신모델 스크리닝 결과와 NP-likeness 점수 모델을 불러온다.

In [ ]:
# 스크리닝 후보(신모델) 불러오기 + NP 모델 로드
HITS = "data/screen_3db_hits_repr.csv"
df = pd.read_csv(HITS)
model = npscorer.readNPModel()          # 천연물다움 점수 모델(수백만 천연물/합성물에서 학습된 통계)
print("후보:", len(df), "개 | NP 모델 로드 완료")

🔎 **코드 뜯어보기 (셀 1)**
- `npscorer.readNPModel()` : 천연물다움 점수를 내는 통계 모델을 메모리에 로드(부분구조 빈도 기반).

### 셀 2 — NP-likeness 점수 계산
후보마다 천연물다움 점수를 매기고 분포를 본다.

In [ ]:
# 각 후보의 NP-likeness 점수 계산 (양수=천연물다움, 음수=합성물다움)
def npscore(smi):
    m = Chem.MolFromSmiles(str(smi))
    return npscorer.scoreMol(m, model) if m else None

df["np_score"] = df["canonical_smiles"].map(npscore)
print("NP-score 분포:")
print(df["np_score"].describe().round(2).to_string())
# 구간별 개수
import numpy as np
for lo, hi, lab in [(-99, 0, "합성물다움(<0)"), (0, 1, "중간(0~1)"), (1, 99, "천연물다움(>=1)")]:
    n = ((df.np_score >= lo) & (df.np_score < hi)).sum()
    print(f"  {lab:16s}: {n}개")

🔎 **코드 뜯어보기 (셀 2)**
- `npscorer.scoreMol(m, model)` : 분자의 부분구조들이 천연물/합성물 중 어디에 흔한지를 합산 → 하나의 점수. 양수일수록 천연물다움.
- `.describe()` : 평균·사분위 등 요약. 구간별 개수로 합성/천연 비율 파악.

### 셀 3 — 최종 정제 & 저장
활성확률(>=0.75)과 천연물다움(np_score>=1.0)을 **동시에** 통과한 후보만 남긴다.

In [ ]:
# 최종 정제: 활성확률 높고(>=0.75) + 천연물다운(np_score>=1.0) 것만
PROB_MIN, NP_MIN = 0.75, 1.0
refined = df[(df.active_prob >= PROB_MIN) & (df.np_score >= NP_MIN)].copy()
refined = refined.sort_values(["np_score", "active_prob"], ascending=False).reset_index(drop=True)
refined.to_csv("data/screen_repr_np_filtered.csv", index=False)

before = (df.active_prob >= PROB_MIN).sum()
print(f"활성확률>={PROB_MIN} 후보 {before}개")
print(f"  → NP-likeness>={NP_MIN} 통과: {len(refined)}개 (합성 열거물 {before-len(refined)}개 제거)")
print("  DB별:", refined.groupby("source").size().to_dict())
print("\n=== 정제 후 상위 15 (천연물다움 순) ===")
print(refined[["np_score", "active_prob", "source", "id", "canonical_smiles"]].head(15).to_string(index=False))
print("\n저장: data/screen_repr_np_filtered.csv")

🔎 **코드 뜯어보기 (셀 3)**
- `df[(df.active_prob>=0.75) & (df.np_score>=1.0)]` : 두 조건 AND. `sort_values(["np_score","active_prob"])` : 천연물다움 우선 정렬. 합성 열거물이 여기서 대거 탈락.